# VIX Market Crash — V5 : XGBoost macro + futures VX1–VX4\n
**Objectif** : prédire une baisse du S&P 500 de 2 % ou plus entre t et t+5, à partir de la courbe du VIX et de cinq marchés supplémentaires : **EUR/USD, or, pétrole WTI, Treasury US 3 ans, rendement corporate high yield US**.

**Sources** : VIX et SPX depuis Google Sheets (identiques à V3) ; DEXUSEU, DCOILWTICO, DGS3 et BAMLH0A0HYM2 depuis FRED ; or via l'ETF GLD (Stooq, proxy). Les données externes sont **décalées d'une séance** après alignement afin d'éviter de présumer qu'elles étaient publiées au moment de la clôture SPX t. Les trous courts (jours fériés) sont comblés, pas les longues périodes sans données.

**Comparaison** : XGBoost VIX seul (baseline V3), XGBoost VIX + macro, XGBoost VIX + macro + probabilité RF OOF. Les modèles utilisent exactement les mêmes dates train/test. Cible prospective avec purge de 5 séances à la frontière train/test. Attention aux fenêtres de rendement qui se chevauchent et à la disponibilité effective des données FRED.

## 0. Imports et paramètres

In [ ]:
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import google.auth
from google.auth.transport.requests import AuthorizedSession
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

STRATEGY_SHEET_ID = '1XKeBA6hcnPLcdPixX5bWBtbDTBWP-MtK7Av8ead0NN0'
CRASH_DAYS = 5
CRASH_FLAG = -0.02
TEST_FRACTION = 0.20
RANDOM_STATE = 42
VIX_FEATURES = ['Slope_1M_3M', 'Slope_3M_6M', 'Slope_6M_1Y',
                'VIX1M_Change', 'VIX3M_Change', 'VIX6M_Change', 'VIX1Y_Change']
MACRO_FEATURES = [
    'VIX1M_Level', 'SPX_Return_Past5D',
    'EURUSD_Level', 'EURUSD_Return_5D',
    'Gold_Return_5D', 'WTI_Return_5D',
    'UST3Y_Yield', 'UST3Y_Change_5D',
    'USHY_Yield', 'USHY_Change_5D', 'HY_minus_UST3Y'
]
ALL_FEATURES = VIX_FEATURES + MACRO_FEATURES

## 1. Google Sheets + FRED : chargement et alignement

In [ ]:
def reload_strategy_excel():
    credentials, _ = google.auth.default(
        scopes=['https://www.googleapis.com/auth/drive.readonly'])
    session = AuthorizedSession(credentials)
    response = session.get(
        f'https://www.googleapis.com/drive/v3/files/{STRATEGY_SHEET_ID}/export',
        params={'mimeType': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'},
        timeout=120)
    response.raise_for_status()
    return BytesIO(response.content)

def align_df_dates(raw, names):
    if raw.shape[1] != 2 * len(names):
        raise ValueError(f'Attendu {2 * len(names)} colonnes Date/Valeur, reçu {raw.shape[1]}')
    aligned = []
    for i, name in enumerate(names):
        part = raw.iloc[:, [2*i, 2*i+1]].copy()
        part.columns = ['Date', name]
        part['Date'] = pd.to_datetime(part['Date'], errors='coerce').dt.normalize()
        part[name] = pd.to_numeric(part[name], errors='coerce')
        part = part.dropna(subset=['Date', name]).drop_duplicates('Date', keep='last')
        aligned.append(part.set_index('Date'))
    return pd.concat(aligned, axis=1, join='inner').sort_index()

strategy_excel = reload_strategy_excel()
data_raw = pd.read_excel(strategy_excel, sheet_name='VIX HISTORIQUE',
                         skiprows=3, usecols=[0,1,2,3,5,6,7,8,9,10])
data = align_df_dates(data_raw, ['SPX','VIX1M','VIX3M','VIX6M','VIX1Y'])
assert len(data) > 100 and not data.isna().any().any()
assert not data.index.has_duplicates and data.index.is_monotonic_increasing

def fred_series(series_id):
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    frame = pd.read_csv(url, na_values=['.'])
    frame.columns = ['Date', series_id]
    frame['Date'] = pd.to_datetime(frame['Date'])
    frame[series_id] = pd.to_numeric(frame[series_id], errors='coerce')
    return frame.set_index('Date')[series_id].sort_index()

def gold_prices():
    # FRED a retiré GOLDAMGBD228NLBM. Utiliser GLD (ETF or, USD)
    # via Stooq comme proxy de prix de l'or ; pas le fixing LBMA.
    url = 'https://stooq.com/q/d/l/?s=gld.us&i=d'
    try:
        frame = pd.read_csv(url)
        if not {'Date', 'Close'}.issubset(frame.columns):
            raise ValueError('CSV Stooq sans colonnes Date / Close')
        frame['Date'] = pd.to_datetime(frame['Date'])
        result = pd.to_numeric(frame.set_index('Date')['Close'], errors='coerce')
        if result.notna().sum() < 500:
            raise ValueError('Historique GLD insuffisant')
        print('Or : proxy ETF GLD (Stooq), cours ajustés selon fournisseur.')
        return result.sort_index()
    except Exception as exc:
        print(f'AVERTISSEMENT : proxy or indisponible ({exc}). '
              'La variable Gold_Return_5D sera exclue des modèles.')
        return pd.Series(dtype=float, name='Gold')

external_raw = pd.DataFrame({
    'EURUSD': fred_series('DEXUSEU'),
    'Gold': gold_prices(),
    'WTI': fred_series('DCOILWTICO'),
    'UST3Y': fred_series('DGS3'),
    'USHY': fred_series('BAMLH0A0HYM2'),
})
# Vérifier la couverture historique : la série or peut s'arrêter avant les autres.
print('Dernière observation disponible par série FRED :')
display(external_raw.apply(lambda col: col.last_valid_index()).to_frame('Dernière date'))
# Alignement sur les séances SPX, remplissage limité aux jours fériés.
# Shift(1) = valeurs externes observées au plus tard à la séance SPX précédente.
external = external_raw.reindex(data.index, method='ffill', limit=3).shift(1)
data = data.join(external)
print('Observations SPX/VIX :', len(data), 'du', data.index.min().date(),
      'au', data.index.max().date())
print('Disponibilité macro :')
display(data[['EURUSD','Gold','WTI','UST3Y','USHY']].notna().sum().to_frame('N séances'))
display(data.tail())
if data['Gold'].notna().sum() < 500:
    print('Or indisponible : la V4 continuera sans la variable Gold_Return_5D.')

## 2. Exploration des données

In [ ]:
display(data.describe().T)
print('Valeurs manquantes :\n', data.isna().sum())
print('Corrélations :')
display(data.corr().round(3))
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
data['SPX'].plot(ax=axes[0], title='S&P 500')
(data['VIX1M'] - data['VIX3M']).plot(ax=axes[1], title='Pente VIX 1M - 3M')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

## 3. Variables VIX + cross-asset, cibles et découpage chronologique

In [ ]:
vix_data = data.copy()
vix_data['Slope_1M_3M'] = data['VIX1M'] - data['VIX3M']
vix_data['Slope_3M_6M'] = data['VIX3M'] - data['VIX6M']
vix_data['Slope_6M_1Y'] = data['VIX6M'] - data['VIX1Y']
for name in ['VIX1M','VIX3M','VIX6M','VIX1Y']:
    vix_data[name + '_Change'] = data[name].diff(CRASH_DAYS)

vix_data['SPX_Return_5D'] = data['SPX'].pct_change(CRASH_DAYS)
vix_data['VIX1M_Level'] = data['VIX1M']
vix_data['SPX_Return_Past5D'] = vix_data['SPX_Return_5D']
vix_data['EURUSD_Level'] = data['EURUSD']
vix_data['EURUSD_Return_5D'] = data['EURUSD'].pct_change(CRASH_DAYS, fill_method=None)
vix_data['Gold_Return_5D'] = data['Gold'].pct_change(CRASH_DAYS, fill_method=None)
vix_data['WTI_Return_5D'] = data['WTI'].pct_change(CRASH_DAYS, fill_method=None)
vix_data['UST3Y_Yield'] = data['UST3Y']
vix_data['UST3Y_Change_5D'] = data['UST3Y'].diff(CRASH_DAYS)
vix_data['USHY_Yield'] = data['USHY']
vix_data['USHY_Change_5D'] = data['USHY'].diff(CRASH_DAYS)
vix_data['HY_minus_UST3Y'] = data['USHY'] - data['UST3Y']

# Exclure uniquement la variable or si le fournisseur alternatif est indisponible.
if vix_data['Gold_Return_5D'].notna().sum() < 500:
    MACRO_FEATURES.remove('Gold_Return_5D')
    ALL_FEATURES = VIX_FEATURES + MACRO_FEATURES
    print('ATTENTION : modèle sans or (données GLD indisponibles).')

# Dates communes à tous les modèles ; n'éliminer que les features manquantes.
vix_data = vix_data.dropna(subset=ALL_FEATURES).copy()
if len(vix_data) < 500:
    raise ValueError(f'Seulement {len(vix_data)} séances communes : vérifier Yahoo/FRED.')
split = int(len(vix_data) * (1 - TEST_FRACTION))
train_dates = vix_data.index[:split]
test_dates = vix_data.index[split:]
thresholds = vix_data.loc[train_dates, VIX_FEATURES].abs().quantile(.75)
vix_data['VIX_Significant_Change'] = vix_data[VIX_FEATURES].abs().gt(thresholds).any(axis=1).astype(int)
vix_data['VIX_Target_T1'] = vix_data['VIX_Significant_Change'].shift(-1)

# IMPORTANT : cible future calculée sur le calendrier SPX original,
# AVANT le filtrage des dates macro (évite de déformer l'horizon de 5 séances).
spx_fwd = data['SPX'].shift(-CRASH_DAYS) / data['SPX'] - 1
vix_data['SPX_Return_Fwd5D'] = spx_fwd.reindex(vix_data.index)
vix_data['Crash_Target_Fwd5D'] = (
    vix_data['SPX_Return_Fwd5D'].le(CRASH_FLAG).astype(float)
    .where(vix_data['SPX_Return_Fwd5D'].notna()))
print('Train :', train_dates[0].date(), '→', train_dates[-1].date())
print('Test :', test_dates[0].date(), '→', test_dates[-1].date())
print('Variables XGB :', len(ALL_FEATURES))
display(thresholds.rename('Seuil 75 % (train seulement)').to_frame())

## 4. Random Forest : test chronologique et prédictions OOF

In [ ]:
# Le dernier point du train est exclu : sa cible T+1 est située dans le test.
rf_train = vix_data.iloc[:split-1].dropna(subset=['VIX_Target_T1'])
rf_test = vix_data.iloc[split:].dropna(subset=['VIX_Target_T1'])
stage1_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
stage1_model.fit(rf_train[VIX_FEATURES], rf_train['VIX_Target_T1'].astype(int))
rf_test_proba = stage1_model.predict_proba(rf_test[VIX_FEATURES])[:,1]
print('Random Forest AUC test :', round(roc_auc_score(rf_test['VIX_Target_T1'], rf_test_proba), 3))
print(classification_report(rf_test['VIX_Target_T1'].astype(int), rf_test_proba >= .5, zero_division=0))

# OOF avec 5 plis chronologiques, et gap=1 pour que les labels T+1 du train
# ne contiennent pas d'observations de la fenêtre de validation.
oof = pd.Series(np.nan, index=vix_data.index, dtype=float)
tscv = TimeSeriesSplit(n_splits=5, gap=1)
for idx_train, idx_val in tscv.split(vix_data.iloc[:split]):
    fit_rows = vix_data.iloc[idx_train].dropna(subset=['VIX_Target_T1'])
    val_rows = vix_data.iloc[idx_val]
    if fit_rows['VIX_Target_T1'].nunique() < 2:
        continue
    fold_rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    fold_rf.fit(fit_rows[VIX_FEATURES], fit_rows['VIX_Target_T1'].astype(int))
    oof.loc[val_rows.index] = fold_rf.predict_proba(val_rows[VIX_FEATURES])[:,1]

vix_data['RF_Probability_T1'] = oof
vix_data.loc[rf_test.index, 'RF_Probability_T1'] = rf_test_proba
# La dernière observation n'a pas de label RF test, mais sa probabilité peut être calculée.
if vix_data.index[-1] in test_dates:
    vix_data.loc[vix_data.index[-1], 'RF_Probability_T1'] = (
        stage1_model.predict_proba(vix_data.iloc[[-1]][VIX_FEATURES])[0,1])
print('Prédictions OOF pour XGB :', oof.notna().sum())

## 5. XGBoost : VIX seul / VIX + macro / VIX + macro + RF

In [ ]:
# Même échantillon pour les 3 modèles.
# Purge temporelle sur le calendrier SPX, pas simplement 5 lignes du jeu filtré.
test_start = test_dates[0]
train_end_safe = data.index[data.index.get_loc(test_start) - CRASH_DAYS]
features_with_rf = ALL_FEATURES + ['RF_Probability_T1']
stage2_train = vix_data.loc[vix_data.index < train_end_safe].dropna(
    subset=features_with_rf + ['Crash_Target_Fwd5D'])
stage2_test = vix_data.loc[vix_data.index >= test_start].dropna(
    subset=features_with_rf + ['Crash_Target_Fwd5D'])
y_train = stage2_train['Crash_Target_Fwd5D'].astype(int)
y_test = stage2_test['Crash_Target_Fwd5D'].astype(int)
if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError('Cible crash : il faut les deux classes en train et test.')
weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

def make_xgb():
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=.05,
                         random_state=RANDOM_STATE, scale_pos_weight=weight,
                         eval_metric='logloss', n_jobs=-1)

feature_sets = {
    'VIX seul (baseline)': VIX_FEATURES,
    'VIX + macro': ALL_FEATURES,
    'VIX + macro + RF': features_with_rf,
}
fitted_models = {}
test_probabilities = {}
for label, cols in feature_sets.items():
    model = make_xgb()
    model.fit(stage2_train[cols], y_train)
    fitted_models[label] = model
    test_probabilities[label] = model.predict_proba(stage2_test[cols])[:, 1]
    print(f'{label} : AUC test = {roc_auc_score(y_test, test_probabilities[label]):.3f}')

auc_base = roc_auc_score(y_test, test_probabilities['VIX seul (baseline)'])
for label in ['VIX + macro', 'VIX + macro + RF']:
    print(f'Gain {label} vs baseline : '
          f'{roc_auc_score(y_test, test_probabilities[label]) - auc_base:+.3f}')
print('\nRapport VIX + macro :')
print(classification_report(y_test, test_probabilities['VIX + macro'] >= .5,
                            zero_division=0))
results_v4 = pd.DataFrame({'Actual_Crash_Fwd5D': y_test,
                           'Realized_SPX_Fwd5D': stage2_test['SPX_Return_Fwd5D']},
                          index=stage2_test.index)
for label, proba in test_probabilities.items():
    results_v4['Probability_' + label] = proba
display(results_v4.tail())

print('\nImportance des variables (gain XGBoost, indicatif) :')
importance = pd.Series(fitted_models['VIX + macro'].feature_importances_,
                       index=ALL_FEATURES).sort_values(ascending=False)
display(importance.to_frame('importance').head(15))

## 6. Backtest indicatif VIX + macro (blocs non chevauchants)
Signal après la clôture t ; entrée hypothétique à la clôture t+1, sortie t+6. Les rendements par blocs de 5 séances ne se chevauchent pas. Comparaison simplifiée, sans frais, slippage ni rémunération du cash.

In [ ]:
bt = results_v4.copy()
bt['Tradable_5D_Return'] = (
    data['SPX'].shift(-(CRASH_DAYS+1)) / data['SPX'].shift(-1) - 1
).reindex(bt.index)
bt = bt.dropna(subset=['Tradable_5D_Return']).iloc[::CRASH_DAYS].copy()
bt['Invested'] = (bt['Probability_VIX + macro'] < .5).astype(int)
bt['Strategy_Return'] = bt['Invested'] * bt['Tradable_5D_Return']
bt['SPX_BuyHold'] = (1 + bt['Tradable_5D_Return']).cumprod()
bt['Strategy'] = (1 + bt['Strategy_Return']).cumprod()
bt[['SPX_BuyHold','Strategy']].plot(figsize=(12,5),
    title='V4 — Backtest indicatif VIX + macro, blocs de 5 séances')
plt.ylabel('Capital normalisé')
plt.grid(True)
plt.show()
print('Blocs :', len(bt))
print('Part investie :', f"{bt['Invested'].mean():.1%}")
print('Performance stratégie :', f"{bt['Strategy'].iloc[-1]-1:.1%}")
print('Performance buy & hold :', f"{bt['SPX_BuyHold'].iloc[-1]-1:.1%}")

## 7. Modèle V5 : XGBoost macro vs macro + VX1–VX4\n\nContrôle des anomalies effectué en amont. Les 14 fortes variations de 2018, 2020 et 2024 sont conservées. Les anciennes cotations sont normalisées dans le constructeur de données. Comparaison sur dates identiques, avec purge de cinq séances, ROC-AUC, average precision et Brier. Aucun résultat de performance n'est présumé avant exécution.\n

In [ ]:
from pathlib import Path
from sklearn.metrics import average_precision_score, brier_score_loss
VX_PATH = Path("vix_futures_cboe/VIX_FUTURES_HISTORIQUE.csv")
ANOMALY_PATH = Path("vix_futures_cboe/VIX_FUTURES_ANOMALIES.csv")
if not VX_PATH.exists():
    raise FileNotFoundError("Lancer python3 build_vix_futures.py dans le dépôt")
vx = pd.read_csv(VX_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
for i in range(1, 5):
    vx[f"VX{i}_Expiry"] = pd.to_datetime(vx[f"VX{i}_Expiry"])
    # Différence quotidienne valide uniquement à expiration identique.
    same_contract = vx[f"VX{i}_Expiry"].eq(vx[f"VX{i}_Expiry"].shift(1))
    vx[f"VX{i}_Change_SameContract"] = vx[f"VX{i}"].diff().where(same_contract)
    vx[f"VX{i}_Pct_SameContract"] = vx[f"VX{i}"].pct_change(fill_method=None).where(same_contract)
vx["VX_Slope_1_2"] = vx["VX2"] - vx["VX1"]
vx["VX_Slope_2_3"] = vx["VX3"] - vx["VX2"]
vx["VX_Slope_3_4"] = vx["VX4"] - vx["VX3"]
vx["VX_Contango_1_2"] = vx["VX2"] / vx["VX1"] - 1
if ANOMALY_PATH.exists():
    anomalies = pd.read_csv(ANOMALY_PATH)
    print("Alertes de qualité:", len(anomalies))
    display(anomalies.groupby(["Series", "Anomaly"]).size().rename("Count").to_frame())
else:
    print("Audit absent : exécuter python3 audit_vix_futures.py")
# Disponibilité à t: décaler les settlements d'une séance par prudence.
VX_MODEL_FEATURES = ["VX1", "VX2", "VX3", "VX4", "VX_Slope_1_2",
                     "VX_Slope_2_3", "VX_Slope_3_4", "VX_Contango_1_2"]
# Exclusion provisoire des cotations manifestement suspectes, sans toucher aux CSV bruts.
# Les observations 2006 ne sont pas utilisées dans le modèle futures (début 2007).
suspect_dates = pd.to_datetime([])  # échelle corrigée dans build_vix_futures.py
vx.loc[vx.index.isin(suspect_dates), VX_MODEL_FEATURES] = np.nan
# Les variations de 2018/2020/2024 sont conservées : elles peuvent être réelles.
vx_features = vx[VX_MODEL_FEATURES].shift(1)
vx_features = vx_features.rename(columns=lambda c: "FUT_" + c)
# Ne pas forward-fill les trous de contrats.
vx_features = vx_features.reindex(data.index)
print("Couverture futures sur les dates SPX disponibles:")
display(vx_features.notna().groupby(vx_features.index.year).mean().round(3).tail(23))

# Entraînement comparatif sur dates communes et cible prospective sans fuite temporelle.
futures_dataset = vix_data.join(vx_features, how="inner")
futures_dataset = futures_dataset.loc["2007-01-01":].dropna(
    subset=ALL_FEATURES + list(vx_features.columns) + ["Crash_Target_Fwd5D"])
if len(futures_dataset) < 1000:
    raise ValueError("Échantillon futures insuffisant : vérifier les dates et l'audit")
n = len(futures_dataset)
boundary = int(n * (1 - TEST_FRACTION))
# Purge des cinq séances d'horizon de la cible.
train = futures_dataset.iloc[:max(0, boundary - CRASH_DAYS)]
test = futures_dataset.iloc[boundary:]
y_train = train["Crash_Target_Fwd5D"].astype(int)
y_test = test["Crash_Target_Fwd5D"].astype(int)
def fit_futures_model(features):
    model = XGBClassifier(n_estimators=250, max_depth=3, learning_rate=.035,
                          subsample=.8, colsample_bytree=.85,
                          eval_metric="logloss", random_state=RANDOM_STATE)
    model.fit(train[features], y_train)
    probabilities = model.predict_proba(test[features])[:, 1]
    auc = roc_auc_score(y_test, probabilities) if y_test.nunique() == 2 else np.nan
    return model, auc, probabilities
base_model, base_auc, base_probs = fit_futures_model(ALL_FEATURES)
futures_model, futures_auc, futures_probs = fit_futures_model(
    ALL_FEATURES + list(vx_features.columns))
print("Comparaison à dates identiques :")
print(f"Macro sans futures : AUC={base_auc:.4f}")
print(f"Macro + VX1–VX4 : AUC={futures_auc:.4f}")
print(f"Macro seule : AP={average_precision_score(y_test, base_probs):.4f}, Brier={brier_score_loss(y_test, base_probs):.4f}")
print(f"Macro + futures : AP={average_precision_score(y_test, futures_probs):.4f}, Brier={brier_score_loss(y_test, futures_probs):.4f}")
print(f"Événements crash : train={int(y_train.sum())}/{len(y_train)}, test={int(y_test.sum())}/{len(y_test)}")
comparison_v5 = pd.DataFrame({"Date":test.index, "Crash_Observed":y_test.values,
    "Prob_Macro":base_probs, "Prob_Macro_Futures":futures_probs})
comparison_v5.to_csv("vix_futures_cboe/VIX_MODEL_COMPARISON_V5.csv", index=False)
print("Résultats exportés : vix_futures_cboe/VIX_MODEL_COMPARISON_V5.csv")
print("Train:", train.index.min().date(), "→", train.index.max().date())
print("Test :", test.index.min().date(), "→", test.index.max().date())
